In [1]:
import json
import pandas as pd


In [2]:
with open(r"C:\Users\Mi\Downloads\export (2).geojson", "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
rows = []

for feature in data["features"]:
    props = feature.get("properties", {})
    geom = feature.get("geometry", {})

    coords = geom.get("coordinates", [None, None])

    rows.append({
        "name": props.get("name"),
        "network": props.get("network"),
        "start_date": props.get("start_date"),
        "longitude": coords[0] if len(coords) > 1 else None,
        "latitude": coords[1] if len(coords) > 1 else None,
    })

df = pd.DataFrame(rows)

print(df.head())

             name                  network  start_date  longitude   latitude
0      Медведково  Московский метрополитен  1978-09-30  37.661550  55.887177
1    Бабушкинская  Московский метрополитен  1978-09-30  37.664110  55.869634
2    Партизанская  Московский метрополитен  1944-01-18  37.750993  55.788503
3     Семёновская  Московский метрополитен  1944-01-18  37.721282  55.783307
4  Филёвский парк  Московский метрополитен  1961-10-13  37.483372  55.739508


In [5]:
df_metro_list = pd.read_csv(r"C:\Users\Mi\Downloads\data-1787903705204.csv")

In [10]:
missing_stations = df.loc[
    ~df['name'].isin(df_metro_list['nearest_metro']),
    'name'
].drop_duplicates()

print(missing_stations)
missing_stations.to_excel('missing_stations.xlsx')

3                   Семёновская
14                   Пионерская
20                 Текстильщики
22                       Выхино
29                  Новогиреево
30                       Перово
34                    Калужская
35              Новые Черёмушки
36                      Ясенево
38                 Черкизовская
49                      Лубянка
75              Цветной бульвар
76               Новоясеневская
79                  Театральная
86                 Добрынинская
89     Бульвар Адмирала Ушакова
98                      Люблино
107             Севастопольская
108                Чертановская
109                       Южная
110                    Пражская
123                Полежаевская
126                 Баррикадная
131        Крестьянская Застава
132           Красногвардейская
137    Бульвар Дмитрия Донского
144                 Университет
146                   Каховская
158                    Коньково
161                 Шипиловская
166      Лермонтовский проспект
191     

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        302 non-null    object 
 1   network     302 non-null    object 
 2   start_date  271 non-null    object 
 3   longitude   302 non-null    float64
 4   latitude    302 non-null    float64
dtypes: float64(2), object(3)
memory usage: 11.9+ KB


In [27]:
df2 = pd.read_csv(r"C:\Users\Mi\Downloads\data-1787587444416.csv")

In [28]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1092 entries, 0 to 1091
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id_zhk  1092 non-null   int64  
 1   lat     1092 non-null   float64
 2   lng     1092 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 25.7 KB


In [29]:
df2.loc[df2[['lat', 'lng']].isna().any(axis=1), 'id_zhk']

Series([], Name: id_zhk, dtype: int64)

In [41]:
import pandas as pd
from sklearn.neighbors import BallTree
import numpy as np

print('Пропуски в станциях:')
print(df[['latitude', 'longitude']].isna().sum())

print('\nПропуски в объектах:')
print(df2[['lat', 'lng']].isna().sum())

# координаты станций
stations = np.deg2rad(
    df[['latitude', 'longitude']].values
)

# координаты объектов
objects = np.deg2rad(
    df2[['lat', 'lng']].values
)

# создаем дерево
tree = BallTree(stations, metric='haversine')

# ищем ближайшую станцию
distances, indexes = tree.query(objects, k=1)

# перевод расстояния в метры
df2['metro_distance_m'] = distances[:, 0] * 6371000

# добавляем название станции
df2['nearest_metro'] = df.iloc[indexes[:, 0]]['name'].values

# добавляем сеть (МЦК/метро)
df2['network'] = df.iloc[indexes[:, 0]]['network'].values

df2['start_date'] = df.iloc[indexes[:, 0]]['start_date'].values


Пропуски в станциях:
latitude     0
longitude    0
dtype: int64

Пропуски в объектах:
lat    0
lng    0
dtype: int64


In [45]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1087 entries, 0 to 1091
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   id_zhk                   1087 non-null   int64   
 1   lat                      1087 non-null   float64 
 2   lng                      1087 non-null   float64 
 3   metro_distance_m         1087 non-null   float64 
 4   nearest_metro            1087 non-null   object  
 5   network                  1087 non-null   object  
 6   metro_distance_category  1087 non-null   category
 7   start_date               1018 non-null   object  
dtypes: category(1), float64(3), int64(1), object(3)
memory usage: 69.2+ KB


In [46]:
df2['metro_distance_category'] = pd.cut(
    df2['metro_distance_m'],
    bins=[0, 500, 1000, 1500, 3000, 1000000],
    labels=[
        'до 500 м',
        '500-1000 м',
        '1-1.5 км',
        '1.5-3 км',
        'более 3 км'
    ]
)

In [48]:
df2[df2['id_zhk'].duplicated(keep=False)].sort_values('id_zhk')

,id_zhk,lat,lng,metro_distance_m,nearest_metro,network,metro_distance_category,start_date


In [49]:
df2 = df2.drop_duplicates(subset='id_zhk', keep='first')

In [50]:
df2.to_csv(r"C:\Users\Mi\Downloads\zk_coords_with_metro.csv", index=False)